# Визуализация данных

© Гошин Е.В., доцент кафедры технической кибернетики, к.т.н., Самарский университет  
© Петров М.В., старший преподаватель кафедры киберфотоники, Самарский университет

# Лекция 2. Преобразование данных в аккуратную (tidy) форму

## Содержание

1. [Введение](#2.1-Введение)
2. [Преобразование значений переменных, выступающих именами столбцов, с помощью `stack`](#22-преобразование-значений-переменных-выступающих-именами-столбцов-с-помощью-stack)
3. [Преобразование значений переменных, выступающих именами столбцов, с помощью `melt`](#23-преобразование-значений-переменных-выступающих-именами-столбцов-с-помощью-melt)
4. [Одновременное использование `stack` для нескольких групп переменных](#24-одновременное-использование-stack-для-нескольких-групп-переменных)
5. [Инвертирование данных, полученных `stack`/`melt`](#25-инвертирование-данных-полученных-stackmelt)
6. [Использование `unstack` после агрегации `groupby`](#26-использование-unstack-после-агрегации-groupby)
7. [Повторение функционала `pivot_table` с помощью агрегации `groupby`](#27-повторение-функционала-pivot_table-с-помощью-агрегации-groupby)
8. [Переименование уровней осей для удобного изменения формы данных](#28-переименование-уровней-осей-для-удобного-изменения-формы-данных)
9. [Приведение к аккуратной форме, когда несколько переменных хранятся в именах столбцов](#29-приведение-к-аккуратной-форме-когда-несколько-переменных-хранятся-в-именах-столбцов)
10. [Приведение к аккуратной форме, когда несколько переменных хранятся в одном столбце](#210-приведение-к-аккуратной-форме-когда-несколько-переменных-хранятся-в-одном-столбце)
11. [Приведение к аккуратной форме, когда в одной ячейке хранится два и более значений](#211-приведение-к-аккуратной-форме-когда-в-одной-ячейке-хранится-два-и-более-значений)
12. [Приведение к аккуратной форме, когда переменные хранятся в именах столбцов и в значениях](#212-приведение-к-аккуратной-форме-когда-переменные-хранятся-в-именах-столбцов-и-в-значениях)
13. [Приведение к аккуратной форме, когда в одной таблице хранятся несколько типов наблюдений](#213-приведение-к-аккуратной-форме-когда-в-одной-таблице-хранятся-несколько-типов-наблюдений)

## 2.1 Введение

Многие реальные наборы данных требуют существенных преобразований перед тем, как переходить к более детальному анализу. В некоторых случаях весь проект может сводиться к форматированию данных таким образом, чтобы их можно было легко обработать.

Существует множество терминов, описывающих процесс структуризации данных; среди специалистов по данным чаще всего используется термин *tidy data*. Его предложил Хэдли Уикем (Hadley Wickham) для описания такой формы представления данных, которая облегчает анализ. В этой лекции будут рассмотрены многие идеи, сформулированные Хэдли, и способы их реализации в `pandas`.

Что такое *tidy data*? Хэдли формулирует три простых руководящих принципа, по которым определяется, является ли набор данных «аккуратным» («упорядоченным»):
- Каждая переменная образует отдельный столбец
- Каждое наблюдение образует отдельную строку
- Каждый тип наблюдений образует отдельную таблицу

Любой набор данных, не удовлетворяющий этим правилам, считается *«messy»* (неаккуратным). Определение станет более ясным, когда мы начнём перестраивать наши данные в tidy-формат, но сейчас нам нужно понимать, что такое переменные, наблюдения и типы наблюдений.

Чтобы интуитивно понять, что такое переменная, полезно различать ***имя переменной*** и ***её значение***. Имена переменных &ndash; это метки (*label*), такие как *gender*, *race*, *salary*, *position*. Значения переменных &ndash; это то, что может изменяться для каждого наблюдения: например, *male/female* для *gender* или *white/black* для *race*. Одно наблюдение &ndash; это совокупность всех значений переменных для одного типа наблюдений. Чтобы понять, что такое тип наблюдений, можно рассмотреть розничный магазин: у него есть данные по каждой транзакции, сотруднику, клиенту, товару и самому магазину. Каждое из этих понятий можно рассматривать как один тип наблюдений, и для каждого потребуется отдельная таблица. Объединение информации о сотрудниках (например, количество отработанных часов) с информацией о клиентах (например, сумма покупки) в одной таблице нарушает этот принцип `tidy`.

Первый шаг к исправлению неаккуратных данных &ndash; научиться распознавать их, и вариантов тут очень много. Хэдли явно выделяет пять наиболее распространённых типов:
- Имена столбцов &ndash; это значения, а не имена переменных
- Несколько переменных сохранены в именах столбцов
- Переменные сохранены и в строках, и в столбцах
- Несколько типов наблюдений сохранены в одной таблице
- Один тип наблюдений сохранён в нескольких таблицах

Важно понимать, что приведение к аккуратной форме обычно не включает изменение значений, заполнение пропусков или какой-либо анализ. Это именно *изменение формы* или *структуры данных* для соответствия принципам `tidy`. Как только данные приведены в правильную форму, дальнейший анализ упрощается.

Обнаружив неаккуратные данные, можно использовать инструменты `pandas` для их структурирования. Основные инструменты &ndash; это методы `DataFrame`: `stack`, `melt`, `unstack` и `pivot`. Более сложные преобразования требуют разбора текста, для чего используется `str`. Вспомогательные методы, такие как `rename`, `rename_axis`, `reset_index` и `set_index`, помогут довести итоговую структуру до нужного вида.

### Наиболее распространённые случаи

- Преобразование значений переменных, выступающих именами столбцов, с помощью `stack`  
- Преобразование значений переменных, выступающих именами столбцов, с помощью `melt`  
- Одновременное использование `stack` для нескольких групп переменных  
- Инвертирование данных, полученных `stack`/`melt`  
- Использование `unstack` после агрегации с помощью `groupby`  
- Повторение функционала `pivot_table` с помощью агрегации `groupby`  
- Переименование уровней осей для удобного изменения формы данных  
- Приведение к аккуратной форме, когда несколько переменных хранятся в именах столбцов  
- Приведение к аккуратной форме, когда несколько переменных хранятся в значениях столбцов  
- Приведение к аккуратной форме, когда в одной ячейке хранится два и более значений  
- Приведение к аккуратной форме, когда переменные хранятся и в именах столбцов, и в значениях  
- Приведение к аккуратной форме, когда в одной таблице хранятся несколько наблюдательных единиц  

Источники:
- [Wickham, H. Tidy data / H. Wickham // Journal of Statistical Software. - 2014. - V. 59(10). - P. 1–23.](http://vita.had.co.nz/papers/tidy-data.pdf)
- [Tidy data](https://tidyr.tidyverse.org/articles/tidy-data.html)
- [Data tidying: Подготовка наборов данных для анализа на конкретных примерах @ Хабр](https://habr.com/ru/articles/248741/)

## 2.2 Преобразование значений переменных, выступающих именами столбцов, с помощью `stack`

Чтобы лучше понять различия между аккуратными и неаккуратными данными, рассмотрим простую таблицу, которую можно представить как в аккуратной форме, так и нет.

In [1]:
import pandas as pd
import numpy as np

In [2]:
state_fruit = pd.read_csv('data/state_fruit.csv', index_col=0)
state_fruit

,Яблоки,Апельсины,Бананы
Самара,12,10,40
Москва,9,7,12
Казань,0,14,190


На первый взгляд ничего «неаккуратного» в таблице нет, и информацию легко воспринимать. Однако согласно принципам `tidy` она неаккуратна: каждое имя столбца &ndash; это значение переменной. Более того, имён переменных в `DataFrame` вообще нет. Один из первых шагов &ndash; определить все переменные. В данном наборе это *state* и *fruit*. Численные данные нигде явно не именованы; их можно обозначить как *weight* или любым другим подходящим названием.

### Постановка задачи

В этом наборе значения переменных выступают именами столбцов. Нам нужно преобразовать имена столбцов в значения одного столбца. Используем метод `stack`, чтобы привести `DataFrame` к аккуратной форме.

### Ключевые этапы

1) Названия штатов (*административно-территориальных единиц*) находятся в индексе `DataFrame`. Они уже расположены вертикально и не требуют переработки. Проблема &ndash; в именах столбцов. `stack` берёт все имена столбцов и преобразует их в вертикальный вид как один уровень индекса.

In [3]:
state_fruit.stack()

Самара  Яблоки        12
        Апельсины     10
        Бананы        40
Москва  Яблоки         9
        Апельсины      7
        Бананы        12
Казань  Яблоки         0
        Апельсины     14
        Бананы       190
dtype: int64

In [4]:
state_fruit.stack().index

MultiIndex([('Самара',    'Яблоки'),
            ('Самара', 'Апельсины'),
            ('Самара',    'Бананы'),
            ('Москва',    'Яблоки'),
            ('Москва', 'Апельсины'),
            ('Москва',    'Бананы'),
            ('Казань',    'Яблоки'),
            ('Казань', 'Апельсины'),
            ('Казань',    'Бананы')],
           )

2) В результате получаем `Series` с многоуровневым индексом (`MultiIndex`). Исходный индекс сдвинут влево, чтобы освободить место для бывших имён столбцов. Одной этой командой мы фактически получили аккуратные данные: каждая переменная &ndash; *state*, *fruit* и *weight* &ndash; представлена вертикально. Применим `reset_index`, чтобы превратить результат в `DataFrame`.

In [5]:
state_fruit_tidy = state_fruit.stack().reset_index()
state_fruit_tidy

,level_0,level_1,0
0,Самара,Яблоки,12
1,Самара,Апельсины,10
2,Самара,Бананы,40
3,Москва,Яблоки,9
4,Москва,Апельсины,7
5,Москва,Бананы,12
6,Казань,Яблоки,0
7,Казань,Апельсины,14
8,Казань,Бананы,190


3) Структура правильная, но имена столбцов неинформативны. Заменим их осмысленными идентификаторами.

In [6]:
state_fruit_tidy.columns = ['state', 'fruit', 'weight']
state_fruit_tidy

,state,fruit,weight
0,Самара,Яблоки,12
1,Самара,Апельсины,10
2,Самара,Бананы,40
3,Москва,Яблоки,9
4,Москва,Апельсины,7
5,Москва,Бананы,12
6,Казань,Яблоки,0
7,Казань,Апельсины,14
8,Казань,Бананы,190


4) Вместо прямого изменения `columns` можно использовать `rename_axis`, чтобы задать имена уровней индекса до `reset_index`.

In [7]:
state_fruit.stack().rename_axis(['state', 'fruit'])

state   fruit    
Самара  Яблоки        12
        Апельсины     10
        Бананы        40
Москва  Яблоки         9
        Апельсины      7
        Бананы        12
Казань  Яблоки         0
        Апельсины     14
        Бананы       190
dtype: int64

5) Можно объединить `reset_index` с параметром `name`, чтобы воспроизвести результат шага 3.

In [8]:
state_fruit.stack().rename_axis(['state', 'fruit']).reset_index(name='weight')

,state,fruit,weight
0,Самара,Яблоки,12
1,Самара,Апельсины,10
2,Самара,Бананы,40
3,Москва,Яблоки,9
4,Москва,Апельсины,7
5,Москва,Бананы,12
6,Казань,Яблоки,0
7,Казань,Апельсины,14
8,Казань,Бананы,190


### Справочная информация

`stack` переносит все имена столбцов во внутренний уровень индекса. Каждое старое имя столбца остаётся связанным со «своими» значениями, будучи спаренным с каждым штатом. В исходном `DataFrame` 3×3 было девять значений &ndash; они стали одним `Series` с тем же количеством элементов. Первая строка исходных данных превратилась в первые три значения `Series`.

После `reset_index` (шаг 2) `pandas` по умолчанию называет столбцы `level_0`, `level_1` и `0`, так как у исходного `Series` два безымянных уровня индекса. Уровни индекса нумеруются снаружи внутрь, начиная с нуля.

Шаг 3 показывает простой способ переименования столбцов: присвоить атрибуту `columns` список имён. Альтернативно можно за один шаг задать имена уровней индекса, «сцепив» `rename_axis` (передаём список имён уровней); при `reset_index` `pandas` использует эти имена как новые имена столбцов. Параметр `name` у `reset_index` задаёт имя столбца для значений `Series`.

У всех `Series` есть атрибут `name`, который можно задать напрямую или через `rename`. Именно он станет именем столбца при использовании `reset_index`.

### Дополнительно

Для корректного применения `stack` необходимо поместить все столбцы, которые **НЕ** нужно преобразовывать, в индекс. Если названия штатов не в индексе, вызов `stack` соберёт и их тоже, превратив всё в один длинный `Series`. Правильный порядок: сначала `set_index` для столбцов, которые не следует преобразовывать, затем `stack`.

In [9]:
state_fruit2 = pd.read_csv('data/state_fruit2.csv')
state_fruit2

,Город,Яблоки,Апельсины,Бананы
0,Самара,12,10,40
1,Москва,9,7,12
2,Казань,0,14,190


In [10]:
state_fruit2.stack()

0  Город        Самара
   Яблоки           12
   Апельсины        10
   Бананы           40
1  Город        Москва
   Яблоки            9
   Апельсины         7
   Бананы           12
2  Город        Казань
   Яблоки            0
   Апельсины        14
   Бананы          190
dtype: object

In [11]:
state_fruit2.set_index('Город').stack()

Город            
Самара  Яблоки        12
        Апельсины     10
        Бананы        40
Москва  Яблоки         9
        Апельсины      7
        Бананы        12
Казань  Яблоки         0
        Апельсины     14
        Бананы       190
dtype: int64

## 2.3 Преобразование значений переменных, выступающих именами столбцов, с помощью `melt`

Как и во многих крупных библиотеках Python, в `pandas` есть несколько способов выполнить одну и ту же задачу, отличающихся читаемостью и производительностью. Метод `DataFrame` `melt` работает подобно `stack`, но предоставляет больше гибкости.

### Постановка задачи

Использовать `melt`, чтобы привести простой `DataFrame` к аккуратной форме, когда значения переменных находятся в именах столбцов.

### Ключевые этапы

1) Считаем набор `state_fruit2` и определим, какие столбцы нужно преобразовывать, а какие &ndash; нет.  

In [12]:
state_fruit2 = pd.read_csv('data/state_fruit2.csv')
state_fruit2

,Город,Яблоки,Апельсины,Бананы
0,Самара,12,10,40
1,Москва,9,7,12
2,Казань,0,14,190


2) Вызовем `melt`, передав соответствующие столбцы параметрам `id_vars` и `value_vars`.

In [13]:
state_fruit2.melt(id_vars=['Город'],
                  value_vars=['Яблоки', 'Апельсины', 'Бананы'])

,Город,variable,value
0,Самара,Яблоки,12
1,Москва,Яблоки,9
2,Казань,Яблоки,0
3,Самара,Апельсины,10
4,Москва,Апельсины,7
5,Казань,Апельсины,14
6,Самара,Бананы,40
7,Москва,Бананы,12
8,Казань,Бананы,190


3) За один шаг сразу получаем аккуратные данные. По умолчанию `melt` называет бывшие имена столбцов `variable`, а соответствующие значения &ndash; `value`. Параметры `var_name` и `value_name` позволяют переименовать эти столбцы.

In [14]:
state_fruit2.melt(id_vars=['Город'],
                  value_vars=['Яблоки', 'Апельсины', 'Бананы'],
                  var_name='Фрукты',
                  value_name='Вес')

,Город,Фрукты,Вес
0,Самара,Яблоки,12
1,Москва,Яблоки,9
2,Казань,Яблоки,0
3,Самара,Апельсины,10
4,Москва,Апельсины,7
5,Казань,Апельсины,14
6,Самара,Бананы,40
7,Москва,Бананы,12
8,Казань,Бананы,190


### Справочная информация

`melt` радикально меняет форму `DataFrame`. Ключевые параметры:
- `id_vars` &ndash; список столбцов, которые нужно сохранить вертикальными;  
- `value_vars` &ndash; список столбцов, которые нужно объединить в единственный столбец.

Значения `id_vars` повторяются для каждого столбца из `value_vars`. Важно: `melt` игнорирует индекс &ndash; он тихо сбрасывается и заменяется на `RangeIndex`. Если в индексе есть значимые значения, сначала необходимо выполнить `reset_index`.

### Дополнительно

Все параметры `melt` необязательны: если хотите поместить **все** значения в один столбец, а их прежние имена &ndash; в другой, достаточно вызова по умолчанию. Часто удобнее задавать только `id_vars`, оставляя `value_vars` неуказанным: тогда «объединяются» все остальные столбцы. Если объединяется один столбец, можно передать его имя строкой, без списка.

In [15]:
state_fruit2.melt()

,variable,value
0,Город,Самара
1,Город,Москва
2,Город,Казань
3,Яблоки,12
4,Яблоки,9
5,Яблоки,0
6,Апельсины,10
7,Апельсины,7
8,Апельсины,14
9,Бананы,40


In [16]:
state_fruit2.melt(id_vars='Город')

,Город,variable,value
0,Самара,Яблоки,12
1,Москва,Яблоки,9
2,Казань,Яблоки,0
3,Самара,Апельсины,10
4,Москва,Апельсины,7
5,Казань,Апельсины,14
6,Самара,Бананы,40
7,Москва,Бананы,12
8,Казань,Бананы,190


## 2.4 Одновременное использование `stack` для нескольких групп переменных

Некоторые наборы содержат несколько групп переменных в именах столбцов, которые нужно одновременно преобразовать в свои столбцы. Рассмотрим пример с набором фильмов: выберем столбцы с именами актёров и соответствующим числом лайков в социальной сети.

In [17]:
movie = pd.read_csv('data/movie.csv')
actor = movie[['movie_title', 'actor_1_name', 'actor_2_name', 'actor_3_name', 
               'actor_1_social_network_likes', 'actor_2_social_network_likes', 'actor_3_social_network_likes']]
actor.head()

,movie_title,actor_1_name,actor_2_name,actor_3_name,actor_1_social_network_likes,actor_2_social_network_likes,actor_3_social_network_likes
0,Avatar,CCH Pounder,Joel David Moore,Wes Studi,1000.0,936.0,855.0
1,Pirates of the Caribbean: At World's End,Johnny Depp,Orlando Bloom,Jack Davenport,40000.0,5000.0,1000.0
2,Spectre,Christoph Waltz,Rory Kinnear,Stephanie Sigman,11000.0,393.0,161.0
3,The Dark Knight Rises,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,27000.0,23000.0,23000.0
4,Star Wars: Episode VII - The Force Awakens,Doug Walker,Rob Walker,NaN,131.0,12.0,NaN


Если определить переменные как название фильма, имя актёра и число лайков в социальной сети, то нужно независимо преобразовать два набора столбцов; одного вызова `stack` или `melt` недостаточно.


### Постановка задачи

Привести таблицу актёров к аккуратной форме, одновременно применив `stack` к именам актёров и числу лайков в социальной сети, с помощью функции `wide_to_long`.

### Ключевые этапы

1) Используем `wide_to_long` для преобразования. Для этого имена столбцов, которые будем преобразовывать, должны оканчиваться цифрой. Сначала определим функцию, меняющую имена столбцов.

In [18]:
def change_col_name(col_name):
    col_name = col_name.replace('_name', '')
    if 'social' in col_name:
        fb_idx = col_name.find('social_network_likes')
        col_name = col_name[:5] + col_name[fb_idx - 1:] + col_name[5:fb_idx - 1]
    return col_name

2) Передадим эту функцию в `rename`, чтобы преобразовать имена столбцов.  

In [19]:
actor2 = actor.rename(columns=change_col_name)
actor2.head()

,movie_title,actor_1,actor_2,actor_3,actor_social_network_likes_1,actor_social_network_likes_2,actor_social_network_likes_3
0,Avatar,CCH Pounder,Joel David Moore,Wes Studi,1000.0,936.0,855.0
1,Pirates of the Caribbean: At World's End,Johnny Depp,Orlando Bloom,Jack Davenport,40000.0,5000.0,1000.0
2,Spectre,Christoph Waltz,Rory Kinnear,Stephanie Sigman,11000.0,393.0,161.0
3,The Dark Knight Rises,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,27000.0,23000.0,23000.0
4,Star Wars: Episode VII - The Force Awakens,Doug Walker,Rob Walker,NaN,131.0,12.0,NaN


In [20]:
actor2.info()

<class 'pandas.DataFrame'>
RangeIndex: 4916 entries, 0 to 4915
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   movie_title                   4916 non-null   str    
 1   actor_1                       4909 non-null   str    
 2   actor_2                       4903 non-null   str    
 3   actor_3                       4893 non-null   str    
 4   actor_social_network_likes_1  4909 non-null   float64
 5   actor_social_network_likes_2  4903 non-null   float64
 6   actor_social_network_likes_3  4893 non-null   float64
dtypes: float64(3), str(4)
memory usage: 269.0 KB


3) Вызовем `wide_to_long`, чтобы одновременно преобразовать группы `actor` и `actor_social_network_likes`.

In [21]:
stubs = ['actor', 'actor_social_network_likes']
actor2_tidy = pd.wide_to_long(actor2, 
                              stubnames=stubs, 
                              i=['movie_title'], 
                              j='actor_num', 
                              sep='_').reset_index()
actor2_tidy.head()

,movie_title,actor_num,actor,actor_social_network_likes
0,Avatar,1,CCH Pounder,1000.0
1,Pirates of the Caribbean: At World's End,1,Johnny Depp,40000.0
2,Spectre,1,Christoph Waltz,11000.0
3,The Dark Knight Rises,1,Tom Hardy,27000.0
4,Star Wars: Episode VII - The Force Awakens,1,Doug Walker,131.0


In [22]:
actor2_tidy.info()

<class 'pandas.DataFrame'>
RangeIndex: 14748 entries, 0 to 14747
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   movie_title                 14748 non-null  str    
 1   actor_num                   14748 non-null  int64  
 2   actor                       14705 non-null  str    
 3   actor_social_network_likes  14705 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 461.0 KB


### Справочная информация

Основной параметр `wide_to_long` &ndash; `stubnames` (список строк). Каждая строка обозначает группу столбцов; все столбцы, начинающиеся с этой строки, преобразуются в один столбец. В нашем примере две группы: `actor` и `actor_social_network_likes`. По умолчанию столбцы в группах должны оканчиваться цифрой &ndash; она становится меткой реструктурированных данных. Если между «заготовкой» и цифрой есть `_`, укажите `sep='_'`.

Имена столбцов изначально не соответствуют шаблону; мы приводим их функцией (например, удаляем суффикс `_name` у актёров и перестраиваем имена с лайками в социальной сети так, чтобы они заканчивались цифрами). `rename` умеет принимать функцию: каждый столбец передаётся в неё по одному.

Кроме того, `wide_to_long` требует уникальную идентификационную переменную `i` (останется вертикальной) и параметр `j` &ndash; имя для числовой метки, снятой с конца исходных имён столбцов. По умолчанию `suffix` &ndash; регулярное выражение `\d+` (одна или более цифр).

### Дополнительно

`wide_to_long` работает, когда все группы имеют одинаковые числовые окончания. Если окончания различаются или это не цифры, всё равно можно применить `wide_to_long`: переименуйте столбцы так, чтобы они оканчивались нужными метками, и измените `suffix`, например на `.*`, чтобы матчить произвольный хвост.

In [23]:
df = pd.read_csv('data/stackme.csv')
df

,State,Country,a1,b2,Test,d,e
0,TX,US,0.45,0.3,Test1,2,6
1,MA,US,0.03,1.2,Test2,9,7
2,ON,CAN,0.70,4.2,Test3,4,2


In [24]:
df2 = df.rename(columns = {'a1':'group1_a1', 'b2':'group1_b2',
                           'd':'group2_a1', 'e':'group2_b2'})
df2

,State,Country,group1_a1,group1_b2,Test,group2_a1,group2_b2
0,TX,US,0.45,0.3,Test1,2,6
1,MA,US,0.03,1.2,Test2,9,7
2,ON,CAN,0.70,4.2,Test3,4,2


In [25]:
pd.wide_to_long(df2, 
                stubnames=['group1', 'group2'], 
                i=['State', 'Country', 'Test'], 
                j='Label', 
                suffix='.+', 
                sep='_')

group1  group2
State Country Test  Label                
TX    US      Test1 a1       0.45       2
                    b2       0.30       6
MA    US      Test2 a1       0.03       9
                    b2       1.20       7
ON    CAN     Test3 a1       0.70       4
                    b2       4.20       2

## 2.5 Инвертирование данных, полученных `stack`/`melt`

У `DataFrame` есть пары методов: `stack`/`unstack` и `melt`/`pivot`, которые преобразуют горизонтальные имена столбцов в вертикальные значения и обратно. Пара `stack`/`unstack` управляет индексами строк/столбцов; `melt`/`pivot` даёт более гибкий выбор столбцов.

### Постановка задачи

Выполнить `stack`/`melt` над набором данных и сразу инвертировать операцию с помощью `unstack`/`pivot`, вернув исходную форму.

### Ключевые этапы

1) Считаем набор `college` с названием учреждения в индексе, оставив только столбцы по расовой структуре бакалавриата. 

In [26]:
usecol_func = lambda x: 'UGDS_' in x or x == 'INSTNM'
college = pd.read_csv('data/college.csv', 
                      index_col='INSTNM', 
                      usecols=usecol_func)
college.head()

,UGDS_WHITE,UGDS_BLACK,UGDS_HISP,UGDS_ASIAN,UGDS_AIAN,UGDS_NHPI,UGDS_2MOR,UGDS_NRA,UGDS_UNKN
INSTNM,,,,,,,,,
Alabama A & M University,0.0333,0.9353,0.0055,0.0019,0.0024,0.0019,0.0000,0.0059,0.0138
University of Alabama at Birmingham,0.5922,0.2600,0.0283,0.0518,0.0022,0.0007,0.0368,0.0179,0.0100
Amridge University,0.2990,0.4192,0.0069,0.0034,0.0000,0.0000,0.0000,0.0000,0.2715
University of Alabama in Huntsville,0.6988,0.1255,0.0382,0.0376,0.0143,0.0002,0.0172,0.0332,0.0350
Alabama State University,0.0158,0.9208,0.0121,0.0019,0.0010,0.0006,0.0098,0.0243,0.0137


2) Применим `stack`, чтобы превратить каждое имя столбца во внутренний уровень индекса.  

In [27]:
college_stacked = college.stack()
college_stacked.head(18)

INSTNM                                         
Alabama A & M University             UGDS_WHITE    0.0333
                                     UGDS_BLACK    0.9353
                                     UGDS_HISP     0.0055
                                     UGDS_ASIAN    0.0019
                                     UGDS_AIAN     0.0024
                                     UGDS_NHPI     0.0019
                                     UGDS_2MOR     0.0000
                                     UGDS_NRA      0.0059
                                     UGDS_UNKN     0.0138
University of Alabama at Birmingham  UGDS_WHITE    0.5922
                                     UGDS_BLACK    0.2600
                                     UGDS_HISP     0.0283
                                     UGDS_ASIAN    0.0518
                                     UGDS_AIAN     0.0022
                                     UGDS_NHPI     0.0007
                                     UGDS_2MOR     0.0368
                        

3) Инвертируем результат методом `unstack` у `Series`.  

In [28]:
college_stacked.unstack().head()

,UGDS_WHITE,UGDS_BLACK,UGDS_HISP,UGDS_ASIAN,UGDS_AIAN,UGDS_NHPI,UGDS_2MOR,UGDS_NRA,UGDS_UNKN
INSTNM,,,,,,,,,
Alabama A & M University,0.0333,0.9353,0.0055,0.0019,0.0024,0.0019,0.0000,0.0059,0.0138
University of Alabama at Birmingham,0.5922,0.2600,0.0283,0.0518,0.0022,0.0007,0.0368,0.0179,0.0100
Amridge University,0.2990,0.4192,0.0069,0.0034,0.0000,0.0000,0.0000,0.0000,0.2715
University of Alabama in Huntsville,0.6988,0.1255,0.0382,0.0376,0.0143,0.0002,0.0172,0.0332,0.0350
Alabama State University,0.0158,0.9208,0.0121,0.0019,0.0010,0.0006,0.0098,0.0243,0.0137


4) Повторим логику с `melt` и `pivot`: прочитаем данные без помещения названия учреждения в индекс.

In [29]:
college2 = pd.read_csv('data/college.csv', 
                       usecols=usecol_func)
college2.head()

,INSTNM,UGDS_WHITE,UGDS_BLACK,UGDS_HISP,UGDS_ASIAN,UGDS_AIAN,UGDS_NHPI,UGDS_2MOR,UGDS_NRA,UGDS_UNKN
0,Alabama A & M University,0.0333,0.9353,0.0055,0.0019,0.0024,0.0019,0.0000,0.0059,0.0138
1,University of Alabama at Birmingham,0.5922,0.2600,0.0283,0.0518,0.0022,0.0007,0.0368,0.0179,0.0100
2,Amridge University,0.2990,0.4192,0.0069,0.0034,0.0000,0.0000,0.0000,0.0000,0.2715
3,University of Alabama in Huntsville,0.6988,0.1255,0.0382,0.0376,0.0143,0.0002,0.0172,0.0332,0.0350
4,Alabama State University,0.0158,0.9208,0.0121,0.0019,0.0010,0.0006,0.0098,0.0243,0.0137


5) Используем `melt`, чтобы объединить все «расовые» столбцы в один.  

In [30]:
college_melted = college2.melt(id_vars='INSTNM', 
                               var_name='Race',
                               value_name='Percentage')
college_melted.head()

,INSTNM,Race,Percentage
0,Alabama A & M University,UGDS_WHITE,0.0333
1,University of Alabama at Birmingham,UGDS_WHITE,0.5922
2,Amridge University,UGDS_WHITE,0.2990
3,University of Alabama in Huntsville,UGDS_WHITE,0.6988
4,Alabama State University,UGDS_WHITE,0.0158


6) Используем `pivot`, чтобы инвертировать результат предыдущего шага. 

In [31]:
melted_inv = college_melted.pivot(index='INSTNM',
                                  columns='Race',
                                  values='Percentage')
melted_inv.head()

Race,UGDS_2MOR,UGDS_AIAN,UGDS_ASIAN,UGDS_BLACK,UGDS_HISP,UGDS_NHPI,UGDS_NRA,UGDS_UNKN,UGDS_WHITE
INSTNM,,,,,,,,,
A & W Healthcare Educators,0.0000,0.0,0.0000,0.9750,0.0250,0.0,0.0000,0.0000,0.0000
A T Still University of Health Sciences,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ABC Beauty Academy,0.0000,0.0,0.9333,0.0333,0.0333,0.0,0.0000,0.0000,0.0000
ABC Beauty College Inc,0.0000,0.0,0.0000,0.6579,0.0526,0.0,0.0000,0.0000,0.2895
AI Miami International University of Art and Design,0.0018,0.0,0.0018,0.0198,0.4773,0.0,0.0025,0.4644,0.0324


7) Чтобы получить точную копию исходной структуры из шага 4, отсортируем строки и столбцы через `.loc`, затем сделаем `reset_index`.

In [32]:
college2_replication = melted_inv.loc[college2['INSTNM'], college2.columns[1:]].reset_index()
college2.equals(college2_replication)

True

### Справочная информация

В шаге 1 демонстрируется гибкость `read_csv`: `usecols` может принимать список столбцов или функцию (на вход &ndash; имя столбца, на выход &ndash; булево). Это помогает экономить память.

`stack` (шаг 2) помещает имена столбцов во внутренний уровень индекса и возвращает `Series`. `unstack` (шаг 3) берёт значения внутреннего уровня индекса и превращает их в имена столбцов.

> Замечание: результат шага 3 не полностью совпадает с шагом 1, так как `stack` по умолчанию отбрасывает целые строки пропусков. Чтобы сохранить их, используйте `dropna=False` в `stack`.

`melt` (шаг 5) объединяет все «расовые» столбцы в один (если `value_vars=None`, объединяются все столбцы, не входящие в `id_vars`). `pivot` (шаг 6) принимает три строковых параметра: `index`, `columns`, `values`. Столбец из `index` становится индексом, значения `columns` &ndash; именами столбцов, значения `values` раскладываются по пересечениям.

Чтобы получить точную копию при `pivot`, отсортируйте строки и столбцы в исходном порядке (см. шаг 7).

### Дополнительно

По умолчанию `unstack` использует внутренний уровень индекса (`level=-1`). Можно развернуть внешний уровень, указав `level=0`. Для чистого транспонирования `DataFrame` не обязательно использовать `stack`/`unstack` &ndash; достаточно `transpose()` или атрибута `T`.

In [33]:
college.stack().unstack(0)

INSTNM,Alabama A & M University,University of Alabama at Birmingham,Amridge University,University of Alabama in Huntsville,Alabama State University,The University of Alabama,Central Alabama Community College,Athens State University,Auburn University at Montgomery,Auburn University,...,Strayer University-North Dallas,Strayer University-San Antonio,Strayer University-Stafford,WestMed College - Merced,Vantage College,SAE Institute of Technology San Francisco,Rasmussen College - Overland Park,National Personal Training Institute of Cleveland,Bay Area Medical Academy - San Jose Satellite Location,Excel Learning Center-San Antonio South
UGDS_WHITE,0.0333,0.5922,0.2990,0.6988,0.0158,0.7825,0.7255,0.7823,0.5328,0.8507,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_BLACK,0.9353,0.2600,0.4192,0.1255,0.9208,0.1119,0.2613,0.1200,0.3376,0.0704,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_HISP,0.0055,0.0283,0.0069,0.0382,0.0121,0.0348,0.0044,0.0191,0.0074,0.0248,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_ASIAN,0.0019,0.0518,0.0034,0.0376,0.0019,0.0106,0.0025,0.0053,0.0221,0.0227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_AIAN,0.0024,0.0022,0.0000,0.0143,0.0010,0.0038,0.0044,0.0157,0.0044,0.0074,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_NHPI,0.0019,0.0007,0.0000,0.0002,0.0006,0.0009,0.0000,0.0010,0.0016,0.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_2MOR,0.0000,0.0368,0.0000,0.0172,0.0098,0.0261,0.0000,0.0174,0.0297,0.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_NRA,0.0059,0.0179,0.0000,0.0332,0.0243,0.0268,0.0000,0.0057,0.0397,0.0100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_UNKN,0.0138,0.0100,0.2715,0.0350,0.0137,0.0026,0.0019,0.0334,0.0246,0.0140,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
college.T

INSTNM,Alabama A & M University,University of Alabama at Birmingham,Amridge University,University of Alabama in Huntsville,Alabama State University,The University of Alabama,Central Alabama Community College,Athens State University,Auburn University at Montgomery,Auburn University,...,Strayer University-North Dallas,Strayer University-San Antonio,Strayer University-Stafford,WestMed College - Merced,Vantage College,SAE Institute of Technology San Francisco,Rasmussen College - Overland Park,National Personal Training Institute of Cleveland,Bay Area Medical Academy - San Jose Satellite Location,Excel Learning Center-San Antonio South
UGDS_WHITE,0.0333,0.5922,0.2990,0.6988,0.0158,0.7825,0.7255,0.7823,0.5328,0.8507,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_BLACK,0.9353,0.2600,0.4192,0.1255,0.9208,0.1119,0.2613,0.1200,0.3376,0.0704,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_HISP,0.0055,0.0283,0.0069,0.0382,0.0121,0.0348,0.0044,0.0191,0.0074,0.0248,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_ASIAN,0.0019,0.0518,0.0034,0.0376,0.0019,0.0106,0.0025,0.0053,0.0221,0.0227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_AIAN,0.0024,0.0022,0.0000,0.0143,0.0010,0.0038,0.0044,0.0157,0.0044,0.0074,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_NHPI,0.0019,0.0007,0.0000,0.0002,0.0006,0.0009,0.0000,0.0010,0.0016,0.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_2MOR,0.0000,0.0368,0.0000,0.0172,0.0098,0.0261,0.0000,0.0174,0.0297,0.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_NRA,0.0059,0.0179,0.0000,0.0332,0.0243,0.0268,0.0000,0.0057,0.0397,0.0100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UGDS_UNKN,0.0138,0.0100,0.2715,0.0350,0.0137,0.0026,0.0019,0.0334,0.0246,0.0140,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2.6 Использование `unstack` после агрегации `groupby`

Группировка по одному столбцу с агрегацией по одному столбцу даёт простой и наглядный результат. При группировке по нескольким столбцам результат может быть менее удобен для восприятия. Поскольку `groupby` по умолчанию помещает уникальные значения группирующих столбцов в индекс, `unstack` помогает переставить данные в более удобный для сравнения вид.

### Постановка задачи

Используя набор `employee`, выполнить агрегацию с группировкой по нескольким столбцам, затем применить `unstack`.

### Ключевые этапы

1) Прочитаем `employee` и найдем среднюю зарплату по расам.

In [35]:
employee = pd.read_csv('data/employee.csv')
employee.head()

,UNIQUE_ID,POSITION_TITLE,DEPARTMENT,BASE_SALARY,RACE,EMPLOYMENT_TYPE,GENDER,EMPLOYMENT_STATUS,HIRE_DATE,JOB_DATE
0,0,ASSISTANT DIRECTOR (EX LVL),Municipal Courts Department,121862.0,Hispanic/Latino,Full Time,Female,Active,2006-06-12,2012-10-13
1,1,LIBRARY ASSISTANT,Library,26125.0,Hispanic/Latino,Full Time,Female,Active,2000-07-19,2010-09-18
2,2,POLICE OFFICER,Houston Police Department-HPD,45279.0,White,Full Time,Male,Active,2015-02-03,2015-02-03
3,3,ENGINEER/OPERATOR,Houston Fire Department (HFD),63166.0,White,Full Time,Male,Active,1982-02-08,1991-05-25
4,4,ELECTRICIAN,General Services Department,56347.0,White,Full Time,Male,Active,1989-06-19,1994-10-22


In [36]:
employee.groupby('RACE')['BASE_SALARY'].mean().astype(int)

RACE
American Indian or Alaskan Native    60272
Asian/Pacific Islander               61660
Black or African American            50137
Hispanic/Latino                      52345
Others                               51278
White                                64419
Name: BASE_SALARY, dtype: int64

2) Теперь найдем среднюю зарплату по расам и полу одновременно. 

In [37]:
agg = employee.groupby(['RACE', 'GENDER'])['BASE_SALARY'].mean().astype(int)
agg

RACE                               GENDER
American Indian or Alaskan Native  Female    60238
                                   Male      60305
Asian/Pacific Islander             Female    63226
                                   Male      61033
Black or African American          Female    48915
                                   Male      51082
Hispanic/Latino                    Female    46503
                                   Male      54782
Others                             Female    63785
                                   Male      38771
White                              Female    66793
                                   Male      63940
Name: BASE_SALARY, dtype: int64

3) Чтобы легче сравнивать мужские и женские зарплаты внутри каждой расы, примените `unstack` к уровню `gender`.

In [38]:
agg.unstack('GENDER')

GENDER,Female,Male
RACE,,
American Indian or Alaskan Native,60238,60305
Asian/Pacific Islander,63226,61033
Black or African American,48915,51082
Hispanic/Latino,46503,54782
Others,63785,38771
White,66793,63940


4) Аналогично примените `unstack` к уровню `race`.

In [39]:
agg.unstack('RACE')

RACE,American Indian or Alaskan Native,Asian/Pacific Islander,Black or African American,Hispanic/Latino,Others,White
GENDER,,,,,,
Female,60238,63226,48915,46503,63785,66793
Male,60305,61033,51082,54782,38771,63940


### Справочная информация

- Шаг 1 &ndash; одна группировка (*RACE*), один агрегируемый столбец (*BASE_SALARY*), одна функция (`mean`).
- Шаг 2 &ndash; группировка по двум измерениям (раса и пол) даёт `Series` с `MultiIndex`, что затрудняет сравнение. `unstack` переносит один из уровней индекса в столбцы. По умолчанию используется внутренний уровень; нужный можно указать параметром `level` (по имени уровня предпочтительнее).

### Дополнительно

Если группирующих и агрегируемых столбцов несколько, результатом сразу будет `DataFrame` (`MultiIndex` в строках и/или столбцах). Можно последовательно применять `unstack` и `stack`, пока не получите желаемую структуру.

In [40]:
agg2 = employee.groupby(['RACE', 'GENDER'])['BASE_SALARY'].agg(['mean', 'max', 'min']).astype(int)
agg2

mean     max    min
RACE                              GENDER                      
American Indian or Alaskan Native Female  60238   98536  26125
                                  Male    60305   81239  26125
Asian/Pacific Islander            Female  63226  130416  26125
                                  Male    61033  163228  27914
Black or African American         Female  48915  150416  24960
                                  Male    51082  275000  26125
Hispanic/Latino                   Female  46503  126115  26125
                                  Male    54782  165216  26104
Others                            Female  63785   63785  63785
                                  Male    38771   38771  38771
White                             Female  66793  178331  27955
                                  Male    63940  210588  26125

In [41]:
agg2.unstack('GENDER')

mean            max            min       
GENDER                            Female   Male  Female    Male Female   Male
RACE                                                                         
American Indian or Alaskan Native  60238  60305   98536   81239  26125  26125
Asian/Pacific Islander             63226  61033  130416  163228  26125  27914
Black or African American          48915  51082  150416  275000  24960  26125
Hispanic/Latino                    46503  54782  126115  165216  26125  26104
Others                             63785  38771   63785   38771  63785  38771
White                              66793  63940  178331  210588  27955  26125

## 2.7 Повторение функционала `pivot_table` с помощью агрегации `groupby`

На первый взгляд `pivot_table` &ndash; уникальный способ анализа данных. Однако его можно воспроизвести с помощью `groupby` и последующего `unstack`.

### Постановка задачи

Используя набор `flights`, создать сводную таблицу `pivot_table`, затем создать её средствами `groupby`.

### Ключевые этапы

1) Прочитаем `flights` и используем `pivot_table`, чтобы получить общее число отменённых рейсов по аэропорту вылета для каждой авиакомпании (задавая `index`, `columns`, `values`, `aggfunc='sum'`, при необходимости `fill_value=0`).

In [42]:
flights = pd.read_csv('data/flights.csv')
flights.head()

,MONTH,DAY,WEEKDAY,AIRLINE,ORG_AIR,DEST_AIR,SCHED_DEP,DEP_DELAY,AIR_TIME,DIST,SCHED_ARR,ARR_DELAY,DIVERTED,CANCELLED
0,1,1,4,WN,LAX,SLC,1625,58.0,94.0,590,1905,65.0,0,0
1,1,1,4,UA,DEN,IAD,823,7.0,154.0,1452,1333,-13.0,0,0
2,1,1,4,MQ,DFW,VPS,1305,36.0,85.0,641,1453,35.0,0,0
3,1,1,4,AA,DFW,DCA,1555,7.0,126.0,1192,1935,-7.0,0,0
4,1,1,4,WN,LAX,MCI,1720,48.0,166.0,1363,2225,39.0,0,0


In [43]:
fp = flights.pivot_table(index='AIRLINE', 
                         columns='ORG_AIR', 
                         values='CANCELLED', 
                         aggfunc='sum',
                         fill_value=0).round(2)
fp.head()

ORG_AIR,ATL,DEN,DFW,IAH,LAS,LAX,MSP,ORD,PHX,SFO
AIRLINE,,,,,,,,,,
AA,3,4,86,3,3,11,3,35,4,2
AS,0,0,0,0,0,0,0,0,0,0
B6,0,0,0,0,0,0,0,0,0,1
DL,28,1,0,0,1,1,4,0,1,2
EV,18,6,27,36,0,0,6,53,0,0


2) Для воспроизведения начнём с `groupby` по всем столбцам, переданным в `index` и `columns`.  

In [44]:
fg = flights.groupby(['AIRLINE', 'ORG_AIR'])['CANCELLED'].sum()
fg.head()

AIRLINE  ORG_AIR
AA       ATL         3
         DEN         4
         DFW        86
         IAH         3
         LAS         3
Name: CANCELLED, dtype: int64

In [45]:
fg.index

MultiIndex([('AA', 'ATL'),
            ('AA', 'DEN'),
            ('AA', 'DFW'),
            ('AA', 'IAH'),
            ('AA', 'LAS'),
            ('AA', 'LAX'),
            ('AA', 'MSP'),
            ('AA', 'ORD'),
            ('AA', 'PHX'),
            ('AA', 'SFO'),
            ...
            ('VX', 'LAX'),
            ('VX', 'ORD'),
            ('VX', 'SFO'),
            ('WN', 'ATL'),
            ('WN', 'DEN'),
            ('WN', 'LAS'),
            ('WN', 'LAX'),
            ('WN', 'MSP'),
            ('WN', 'PHX'),
            ('WN', 'SFO')],
           names=['AIRLINE', 'ORG_AIR'], length=114)

3) Применим `unstack`, чтобы перенести уровень `ORG_AIR` из индекса в имена столбцов; при необходимости используем `fill_value=0`.

In [46]:
fg_unstack = fg.unstack('ORG_AIR', fill_value=0)
fg_unstack.head()

ORG_AIR,ATL,DEN,DFW,IAH,LAS,LAX,MSP,ORD,PHX,SFO
AIRLINE,,,,,,,,,,
AA,3,4,86,3,3,11,3,35,4,2
AS,0,0,0,0,0,0,0,0,0,0
B6,0,0,0,0,0,0,0,0,0,1
DL,28,1,0,0,1,1,4,0,1,2
EV,18,6,27,36,0,0,6,53,0,0


In [47]:
fp.equals(fg_unstack)

True

### Справочная информация

`pivot_table` фактически агрегирует по пересечениям уникальных комбинаций столбцов из `index` и `columns`. Воспроизведение: сгруппировать по тем же столбцам, агрегировать `values`, затем `unstack` нужный уровень индекса в столбцы.

### Дополнительно

In [48]:
fp2 = flights.pivot_table(index=['AIRLINE', 'MONTH'],
                          columns=['ORG_AIR', 'CANCELLED'],
                          values=['DEP_DELAY', 'DIST'],
                          aggfunc=[np.mean, np.sum],
                          fill_value=0)
fp2.head()

mean                                                       \
              DEP_DELAY                                                        
ORG_AIR             ATL             DEN             DFW             IAH        
CANCELLED             0    1          0    1          0    1          0    1   
AIRLINE MONTH                                                                  
AA      1     -3.250000  0.0   7.062500  0.0  11.977591 -3.0   9.750000  0.0   
        2     -3.000000  0.0   5.461538  0.0   8.756579  0.0   1.000000  0.0   
        3     -0.166667  0.0   7.666667  0.0  15.383784  0.0  10.900000  0.0   
        4      0.071429  0.0  20.266667  0.0  10.501493  0.0   6.933333  0.0   
        5      5.777778  0.0  23.466667  0.0  16.798780  0.0   3.055556  0.0   

                               ...     sum                                \
                               ...    DIST                                 
ORG_AIR              LAS       ...     LAX          MSP        ORD         
CANCELLED              0    1  ...       0     1      0  1       0     1   
AIRLINE MONTH                  ...                                         
AA      1      32.375000  0.0  ...  135921  2475   7281  0  129334     0   
        2      -3.055556  0.0  ...  113483  5454   5040  0  120572  5398   
        3      12.074074  0.0  ...  131836  1744  14471  0  127072   802   
        4      27.241379  0.0  ...  170285     0   4541  0  152154  4718   
        5       2.818182  0.0  ...  167484     0   6298  0  110864  1999   

                                        
                                        
ORG_AIR          PHX         SFO        
CANCELLED          0    1      0     1  
AIRLINE MONTH                           
AA      1      21018    0  33483     0  
        2      17049  868  32110  2586  
        3      25770    0  43580     0  
        4      17727    0  51054     0  
        5      11164    0  40233     0  

[5 rows x 80 columns]

In [49]:
flights.groupby(['AIRLINE', 'MONTH', 'ORG_AIR', 'CANCELLED'])[['DEP_DELAY', 'DIST']] \
       .agg(['mean', 'sum']) \
       .unstack(['ORG_AIR', 'CANCELLED'], fill_value=0) \
       .swaplevel(0, 1, axis='columns') \
       .head()

mean                                                       \
              DEP_DELAY                                                        
ORG_AIR             ATL             DEN             DFW             IAH        
CANCELLED             0    1          0    1          0    1          0    1   
AIRLINE MONTH                                                                  
AA      1     -3.250000  0.0   7.062500  0.0  11.977591 -3.0   9.750000  0.0   
        2     -3.000000  NaN   5.461538  NaN   8.756579  NaN   1.000000  NaN   
        3     -0.166667  NaN   7.666667  0.0  15.383784  NaN  10.900000  0.0   
        4      0.071429  0.0  20.266667  0.0  10.501493  NaN   6.933333  0.0   
        5      5.777778  0.0  23.466667  NaN  16.798780  NaN   3.055556  NaN   

                               ...     sum                                \
                               ...    DIST                                 
ORG_AIR              LAS       ...     LAX          MSP        ORD         
CANCELLED              0    1  ...       0     1      0  1       0     1   
AIRLINE MONTH                  ...                                         
AA      1      32.375000  0.0  ...  135921  2475   7281  0  129334     0   
        2      -3.055556  NaN  ...  113483  5454   5040  0  120572  5398   
        3      12.074074  0.0  ...  131836  1744  14471  0  127072   802   
        4      27.241379  0.0  ...  170285     0   4541  0  152154  4718   
        5       2.818182  0.0  ...  167484     0   6298  0  110864  1999   

                                        
                                        
ORG_AIR          PHX         SFO        
CANCELLED          0    1      0     1  
AIRLINE MONTH                           
AA      1      21018    0  33483     0  
        2      17049  868  32110  2586  
        3      25770    0  43580     0  
        4      17727    0  51054     0  
        5      11164    0  40233     0  

[5 rows x 80 columns]

## 2.8 Переименование уровней осей для удобного изменения формы данных

Преобразования с помощью `stack`/`unstack` проще, когда у каждого уровня осей (индекса/столбцов) есть имя. `Pandas` позволяет ссылаться на уровень по позиции или по имени; лучше использовать имена (явный способ предпочтительнее неявного).

## Постановка задачи

Задать имена уровням и в явном виде управлять структурой данных через `stack`/`unstack`.

### Ключевые этапы

1) Считаем `college` и посчитаем сводные показатели по численности бакалавриата и баллам `SAT` по математике по учреждениям и религиозной принадлежности.

In [50]:
college = pd.read_csv('data/college.csv')
college

,INSTNM,CITY,STABBR,HBCU,MENONLY,WOMENONLY,RELAFFIL,SATVRMID,SATMTMID,DISTANCEONLY,...,UGDS_2MOR,UGDS_NRA,UGDS_UNKN,PPTUG_EF,CURROPER,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
0,Alabama A & M University,Normal,AL,1.0,0.0,0.0,0,424.0,420.0,0.0,...,0.0000,0.0059,0.0138,0.0656,1,0.7356,0.8284,0.1049,30300,33888
1,University of Alabama at Birmingham,Birmingham,AL,0.0,0.0,0.0,0,570.0,565.0,0.0,...,0.0368,0.0179,0.0100,0.2607,1,0.3460,0.5214,0.2422,39700,21941.5
2,Amridge University,Montgomery,AL,0.0,0.0,0.0,1,NaN,NaN,1.0,...,0.0000,0.0000,0.2715,0.4536,1,0.6801,0.7795,0.8540,40100,23370
3,University of Alabama in Huntsville,Huntsville,AL,0.0,0.0,0.0,0,595.0,590.0,0.0,...,0.0172,0.0332,0.0350,0.2146,1,0.3072,0.4596,0.2640,45500,24097
4,Alabama State University,Montgomery,AL,1.0,0.0,0.0,0,425.0,430.0,0.0,...,0.0098,0.0243,0.0137,0.0892,1,0.7347,0.7554,0.1270,26600,33118.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7530,SAE Institute of Technology San Francisco,Emeryville,CA,NaN,NaN,NaN,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,9500
7531,Rasmussen College - Overland Park,Overland Park,KS,NaN,NaN,NaN,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,21163
7532,National Personal Training Institute of Cleveland,Highland Heights,OH,NaN,NaN,NaN,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,6333
7533,Bay Area Medical Academy - San Jose Satellite ...,San Jose,CA,NaN,NaN,NaN,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,PrivacySuppressed


In [51]:
cg = college.groupby(['STABBR', 'RELAFFIL'])[['UGDS', 'SATMTMID']] \
            .agg(['count', 'min', 'max']).head(6)
cg

UGDS                 SATMTMID              
                count    min      max    count    min    max
STABBR RELAFFIL                                             
AK     0            7  109.0  12865.0        0    NaN    NaN
       1            3   27.0    275.0        1  503.0  503.0
AL     0           71   12.0  29851.0       13  420.0  590.0
       1           18   13.0   3033.0        8  400.0  560.0
AR     0           68   18.0  21405.0        9  427.0  565.0
       1           14   20.0   4485.0        7  495.0  600.0

2) Уровни индекса имеют имена (бывшие имена столбцов). Для столбцов без имён &ndash; зададим имена с помощью `rename_axis`.

In [52]:
cg = cg.rename_axis(['AGG_COLS', 'AGG_FUNCS'], axis='columns')
cg

AGG_COLS         UGDS                 SATMTMID              
AGG_FUNCS       count    min      max    count    min    max
STABBR RELAFFIL                                             
AK     0            7  109.0  12865.0        0    NaN    NaN
       1            3   27.0    275.0        1  503.0  503.0
AL     0           71   12.0  29851.0       13  420.0  590.0
       1           18   13.0   3033.0        8  400.0  560.0
AR     0           68   18.0  21405.0        9  427.0  565.0
       1           14   20.0   4485.0        7  495.0  600.0

3) Применим `stack`, чтобы переместить уровень столбцов `AGG_FUNCS` в уровень индекса.  

In [53]:
cg.stack('AGG_FUNCS').head(6)

AGG_COLS                      UGDS  SATMTMID
STABBR RELAFFIL AGG_FUNCS                   
AK     0        count          7.0       0.0
                min          109.0       NaN
                max        12865.0       NaN
       1        count          3.0       1.0
                min           27.0     503.0
                max          275.0     503.0

4) По умолчанию новый уровень становится внутренним &ndash; используем `swaplevel`, чтобы поменять уровни местами.

In [54]:
cg.stack('AGG_FUNCS').swaplevel('AGG_FUNCS', 'STABBR', axis='index').head(6)

,,AGG_COLS,UGDS,SATMTMID
AGG_FUNCS,RELAFFIL,STABBR,,
count,0,AK,7.0,0.0
min,0,AK,109.0,NaN
max,0,AK,12865.0,NaN
count,1,AK,3.0,1.0
min,1,AK,27.0,503.0
max,1,AK,275.0,503.0


5) Отсортируем уровни через `sort_index`.  

In [55]:
cg.stack('AGG_FUNCS') \
  .swaplevel('AGG_FUNCS', 'STABBR', axis='index') \
  .sort_index(level='RELAFFIL', axis='index') \
  .sort_index(level='AGG_COLS', axis='columns').head(6)

AGG_COLS                   SATMTMID     UGDS
AGG_FUNCS RELAFFIL STABBR                   
count     0        AK           0.0      7.0
                   AL          13.0     71.0
                   AR           9.0     68.0
max       0        AK           NaN  12865.0
                   AL         590.0  29851.0
                   AR         565.0  21405.0

6) Комбинируем `stack` для одних уровней и `unstack` для других в одной цепочке операций.  

In [56]:
cg.stack('AGG_FUNCS').unstack(['RELAFFIL', 'STABBR'])

AGG_COLS      UGDS                                          SATMTMID         \
RELAFFIL         0      1        0       1        0       1        0      1   
STABBR          AK     AK       AL      AL       AR      AR       AK     AK   
AGG_FUNCS                                                                     
count          7.0    3.0     71.0    18.0     68.0    14.0      0.0    1.0   
min          109.0   27.0     12.0    13.0     18.0    20.0      NaN  503.0   
max        12865.0  275.0  29851.0  3033.0  21405.0  4485.0      NaN  503.0   

AGG_COLS                               
RELAFFIL       0      1      0      1  
STABBR        AL     AL     AR     AR  
AGG_FUNCS                              
count       13.0    8.0    9.0    7.0  
min        420.0  400.0  427.0  495.0  
max        590.0  560.0  565.0  600.0

7) Чтобы получить `Series`, можно перенести все уровни столбцов в индекс одновременно.

In [57]:
cg.stack(['AGG_FUNCS', 'AGG_COLS']).head(12)

STABBR  RELAFFIL  AGG_FUNCS  AGG_COLS
AK      0         count      UGDS            7.0
                  min        UGDS          109.0
                  max        UGDS        12865.0
                  count      SATMTMID        0.0
                  min        SATMTMID        NaN
                  max        SATMTMID        NaN
        1         count      UGDS            3.0
                  min        UGDS           27.0
                  max        UGDS          275.0
                  count      SATMTMID        1.0
                  min        SATMTMID      503.0
                  max        SATMTMID      503.0
dtype: float64

In [58]:
cg.stack(['AGG_FUNCS', 'AGG_COLS']).index

MultiIndex([('AK', 0, 'count',     'UGDS'),
            ('AK', 0,   'min',     'UGDS'),
            ('AK', 0,   'max',     'UGDS'),
            ('AK', 0, 'count', 'SATMTMID'),
            ('AK', 0,   'min', 'SATMTMID'),
            ('AK', 0,   'max', 'SATMTMID'),
            ('AK', 1, 'count',     'UGDS'),
            ('AK', 1,   'min',     'UGDS'),
            ('AK', 1,   'max',     'UGDS'),
            ('AK', 1, 'count', 'SATMTMID'),
            ('AK', 1,   'min', 'SATMTMID'),
            ('AK', 1,   'max', 'SATMTMID'),
            ('AL', 0, 'count',     'UGDS'),
            ('AL', 0,   'min',     'UGDS'),
            ('AL', 0,   'max',     'UGDS'),
            ('AL', 0, 'count', 'SATMTMID'),
            ('AL', 0,   'min', 'SATMTMID'),
            ('AL', 0,   'max', 'SATMTMID'),
            ('AL', 1, 'count',     'UGDS'),
            ('AL', 1,   'min',     'UGDS'),
            ('AL', 1,   'max',     'UGDS'),
            ('AL', 1, 'count', 'SATMTMID'),
            ('AL', 1,   'min', '

### Справочная информация

`rename_axis` способен менять как имена уровней (если передать список/скаляр), так и значения уровней (если передать словарь/функцию). После именования уровней управление формой данных становится явным: `stack` переносит указанный уровень столбцов в индекс, `swaplevel` меняет порядок уровней, `sort_index` сортирует значения уровней.

### Дополнительно

Если хотите избавиться от имён уровней для уменьшения «визуального шума», установите их в `None`.

In [59]:
cg.rename_axis([None, None], axis='index').rename_axis([None, None], axis='columns')

UGDS                 SATMTMID              
     count    min      max    count    min    max
AK 0     7  109.0  12865.0        0    NaN    NaN
   1     3   27.0    275.0        1  503.0  503.0
AL 0    71   12.0  29851.0       13  420.0  590.0
   1    18   13.0   3033.0        8  400.0  560.0
AR 0    68   18.0  21405.0        9  427.0  565.0
   1    14   20.0   4485.0        7  495.0  600.0

## 2.9 Приведение к аккуратной форме, когда несколько переменных хранятся в именах столбцов

Один из частых видов неаккуратных данных &ndash; когда имена столбцов содержат несколько переменных (например, пол и возраст, склеенные вместе). Чтобы привести такой набор к аккуратной форме, манипулируем именами столбцов с помощью аксессора `str`.

### Постановка задачи

Идентифицировать переменные (часть из них склеена в именах столбцов), затем преобразовать текст, извлекая корректные значения переменных.

### Ключевые этапы

1) Прочитаем набор по мужской тяжёлой атлетике и определим переменные.

In [60]:
weightlifting = pd.read_csv('data/weightlifting_men.csv')
weightlifting

,Weight Category,M35 35-39,M40 40-44,M45 45-49,M50 50-54,M55 55-59,M60 60-64,M65 65-69,M70 70-74,M75 75-79,M80 80+
0,56,137,130,125,115,102,92,80,67,62,55
1,62,152,145,137,127,112,102,90,75,67,57
2,69,167,160,150,140,125,112,97,82,75,60
3,77,182,172,165,150,135,122,107,90,82,65
4,85,192,182,175,160,142,130,112,95,87,70
5,94,202,192,182,167,150,137,120,100,90,75
6,105,210,200,190,175,157,142,122,102,95,80
7,105+,217,207,197,182,165,150,127,107,100,85


2) Переменные: весовая категория, категория «пол/возраст», квалификационный итог. Пол и возраст склеены. Сначала используем `melt`, чтобы перенести имена столбцов «возраст/пол» в один вертикальный столбец.  

In [61]:
wl_melt = weightlifting.melt(id_vars='Weight Category', 
                             var_name='sex_age', 
                             value_name='Qual Total')
wl_melt.head()

,Weight Category,sex_age,Qual Total
0,56,M35 35-39,137
1,62,M35 35-39,152
2,69,M35 35-39,167
3,77,M35 35-39,182
4,85,M35 35-39,192


3) Выберем столбец `sex_age` и применим `str.split` для разделения на два столбца.  

In [62]:
sex_age = wl_melt['sex_age'].str.split(expand=True)
sex_age.head()

,0,1
0,M35,35-39
1,M35,35-39
2,M35,35-39
3,M35,35-39
4,M35,35-39


4) Переименуем полученные столбцы в осмысленные имена.  

In [63]:
sex_age.columns = ['Sex', 'Age Group']
sex_age.head()

,Sex,Age Group
0,M35,35-39
1,M35,35-39
2,M35,35-39
3,M35,35-39
4,M35,35-39


5) Используем индексацию после `str`, чтобы взять первый символ из столбца `Sex`.  

In [64]:
sex_age['Sex'] = sex_age['Sex'].str[0]
sex_age.head()

,Sex,Age Group
0,M,35-39
1,M,35-39
2,M,35-39
3,M,35-39
4,M,35-39


6) Объединим результат с `wl_melt` через `pd.concat` (горизонтально), чтобы получить аккуратный набор.  

In [65]:
wl_cat_total = wl_melt[['Weight Category', 'Qual Total']]
wl_tidy = pd.concat([sex_age, wl_cat_total], axis='columns')
wl_tidy.head()

,Sex,Age Group,Weight Category,Qual Total
0,M,35-39,56,137
1,M,35-39,62,152
2,M,35-39,69,167
3,M,35-39,77,182
4,M,35-39,85,192


7) Альтернативная цепочка даёт тот же результат (вариант записи операций).

In [66]:
cols = ['Weight Category', 'Qual Total']
sex_age[cols] = wl_melt[cols]
sex_age

,Sex,Age Group,Weight Category,Qual Total
0,M,35-39,56,137
1,M,35-39,62,152
2,M,35-39,69,167
3,M,35-39,77,182
4,M,35-39,85,192
...,...,...,...,...
75,M,80+,77,65
76,M,80+,85,70
77,M,80+,94,75
78,M,80+,105,80


### Справочная информация

Когда переменные «зашиты» в именах столбцов, используйте `melt` (или `stack`). Переменная «Весовая категория» уже на месте &ndash; передаём её в `id_vars`. Столбец `sex_age` нужно распарсить. `str.split` по умолчанию делит по пробелу; можно задать строку или регулярное выражение (`pat`). При `expand=True` каждая часть попадает в отдельный столбец. Индексация через `str[...]` позволяет извлекать подстроки (например, первый символ для пола). Объединяем части через `concat`.

### Дополнительно

Можно обойтись без `split`, используя `assign` и `str.extract` с регулярным выражением и группами захвата. Пример: извлечь возрастной интервал вида `\d{2}[+-](?:\d{2})?`. После формирования аккуратных столбцов исходный `sex_age` удаляется. Результаты, полученные разными путями, можно сравнить на эквивалентность.

In [67]:
age_group = wl_melt.sex_age.str.extract('(\\d{2}[-+](?:\\d{2})?)', expand=False)
sex = wl_melt.sex_age.str[0]
new_cols = {'Sex': sex, 
            'Age Group': age_group}

In [68]:
wl_tidy2 = wl_melt.assign(**new_cols).drop('sex_age', axis='columns')
wl_tidy2.head()

,Weight Category,Qual Total,Sex,Age Group
0,56,137,M,35-39
1,62,152,M,35-39
2,69,167,M,35-39
3,77,182,M,35-39
4,85,192,M,35-39


In [69]:
wl_tidy2.sort_index(axis=1).equals(wl_tidy.sort_index(axis=1))

True

## 2.10 Приведение к аккуратной форме, когда несколько переменных хранятся в одном столбце

В аккуратных наборах на каждую переменную приходится отдельный столбец. Иногда несколько имён переменных помещают в один столбец, а соответствующее значение &ndash; в другой. Тогда каждая логическая запись «растянута» по нескольким строкам.

### Постановка задачи

Определить столбец с неправильно структурированными переменными и использовать преобразования, чтобы получить аккуратные данные.

### Ключевые этапы

1) Прочитаем набор по проверкам ресторанов и преобразуем тип столбца `Date` в `datetime64`.

In [70]:
inspections = pd.read_csv('data/restaurant_inspections.csv', parse_dates=['Date'])
inspections.head(10)

,Name,Date,Info,Value
0,E & E Grill House,2017-08-08,Borough,MANHATTAN
1,E & E Grill House,2017-08-08,Cuisine,American
2,E & E Grill House,2017-08-08,Description,Non-food contact surface improperly constructe...
3,E & E Grill House,2017-08-08,Grade,A
4,E & E Grill House,2017-08-08,Score,9.0
5,PIZZA WAGON,2017-04-12,Borough,BROOKLYN
6,PIZZA WAGON,2017-04-12,Cuisine,Pizza
7,PIZZA WAGON,2017-04-12,Description,"Food contact surface not properly washed, rins..."
8,PIZZA WAGON,2017-04-12,Grade,A
9,PIZZA WAGON,2017-04-12,Score,10.0


2) Столбцы `Name` и `Date` &ndash; корректные переменные. Столбец `Info` фактически хранит пять разных переменных: `Borough`, `Cuisine`, `Description`, `Grade`, `Score`. Используем `pivot`, чтобы оставить `Name` и `Date` вертикально, создать новые столбцы из значений `Info` и использовать столбец `Value` как значения пересечений.  

In [71]:
inspections.pivot(index=['Name', 'Date'], columns='Info', values='Value')

,Info,Borough,Cuisine,Description,Grade,Score
Name,Date,,,,,
3 STAR JUICE CENTER,2017-05-10,BROOKLYN,"Juice, Smoothies, Fruit Salads",Facility not vermin proof. Harborage or condit...,A,12.0
A & L PIZZA RESTAURANT,2017-08-22,BROOKLYN,Pizza,Facility not vermin proof. Harborage or condit...,A,9.0
AKSARAY TURKISH CAFE AND RESTAURANT,2017-07-25,BROOKLYN,Turkish,Plumbing not properly installed or maintained;...,A,13.0
ANTOJITOS DELI FOOD,2017-06-01,BROOKLYN,"Latin (Cuban, Dominican, Puerto Rican, South &...",Live roaches present in facility's food and/or...,A,10.0
BANGIA,2017-06-16,MANHATTAN,Korean,Covered garbage receptacle not provided or ina...,A,9.0
...,...,...,...,...,...,...
VALL'S PIZZERIA,2017-03-15,STATEN ISLAND,Pizza/Italian,Wiping cloths soiled or not stored in sanitizi...,A,9.0
VIP GRILL,2017-06-12,BROOKLYN,Jewish/Kosher,Hot food item not held at or above 140Âº F.,A,10.0
WAHIZZA,2017-04-13,MANHATTAN,Pizza,"No facilities available to wash, rinse and san...",A,10.0


3) Альтернативный вариант: перенесём `Name`, `Date` и `Info` в индекс через `set_index`.  

In [72]:
inspections.set_index(['Name','Date', 'Info']).head(10)

Value
Name              Date       Info                                                          
E & E Grill House 2017-08-08 Borough                                              MANHATTAN
                             Cuisine                                               American
                             Description  Non-food contact surface improperly constructe...
                             Grade                                                        A
                             Score                                                      9.0
PIZZA WAGON       2017-04-12 Borough                                               BROOKLYN
                             Cuisine                                                  Pizza
                             Description  Food contact surface not properly washed, rins...
                             Grade                                                        A
                             Score                                                     10.0

4) Применим `unstack`, чтобы «разложить» значения `Info` по столбцам.  

In [73]:
inspections.set_index(['Name','Date', 'Info']).unstack('Info').head()

Value  \
Info                                              Borough   
Name                                Date                    
3 STAR JUICE CENTER                 2017-05-10   BROOKLYN   
A & L PIZZA RESTAURANT              2017-08-22   BROOKLYN   
AKSARAY TURKISH CAFE AND RESTAURANT 2017-07-25   BROOKLYN   
ANTOJITOS DELI FOOD                 2017-06-01   BROOKLYN   
BANGIA                              2017-06-16  MANHATTAN   

                                                                                                   \
Info                                                                                      Cuisine   
Name                                Date                                                            
3 STAR JUICE CENTER                 2017-05-10                     Juice, Smoothies, Fruit Salads   
A & L PIZZA RESTAURANT              2017-08-22                                              Pizza   
AKSARAY TURKISH CAFE AND RESTAURANT 2017-07-25                                            Turkish   
ANTOJITOS DELI FOOD                 2017-06-01  Latin (Cuban, Dominican, Puerto Rican, South &...   
BANGIA                              2017-06-16                                             Korean   

                                                                                                   \
Info                                                                                  Description   
Name                                Date                                                            
3 STAR JUICE CENTER                 2017-05-10  Facility not vermin proof. Harborage or condit...   
A & L PIZZA RESTAURANT              2017-08-22  Facility not vermin proof. Harborage or condit...   
AKSARAY TURKISH CAFE AND RESTAURANT 2017-07-25  Plumbing not properly installed or maintained;...   
ANTOJITOS DELI FOOD                 2017-06-01  Live roaches present in facility's food and/or...   
BANGIA                              2017-06-16  Covered garbage receptacle not provided or ina...   

                                                            
Info                                           Grade Score  
Name                                Date                    
3 STAR JUICE CENTER                 2017-05-10     A  12.0  
A & L PIZZA RESTAURANT              2017-08-22     A   9.0  
AKSARAY TURKISH CAFE AND RESTAURANT 2017-07-25     A  13.0  
ANTOJITOS DELI FOOD                 2017-06-01     A  10.0  
BANGIA                              2017-06-16     A   9.0

5) Вернём уровни индекса в столбцы через `reset_index`.  

In [74]:
insp_tidy = inspections.set_index(['Name','Date', 'Info']) \
                       .unstack('Info') \
                       .reset_index(col_level=-1)
insp_tidy.head()

Value  \
Info                                 Name       Date    Borough   
0                     3 STAR JUICE CENTER 2017-05-10   BROOKLYN   
1                  A & L PIZZA RESTAURANT 2017-08-22   BROOKLYN   
2     AKSARAY TURKISH CAFE AND RESTAURANT 2017-07-25   BROOKLYN   
3                     ANTOJITOS DELI FOOD 2017-06-01   BROOKLYN   
4                                  BANGIA 2017-06-16  MANHATTAN   

                                                         \
Info                                            Cuisine   
0                        Juice, Smoothies, Fruit Salads   
1                                                 Pizza   
2                                               Turkish   
3     Latin (Cuban, Dominican, Puerto Rican, South &...   
4                                                Korean   

                                                                     
Info                                        Description Grade Score  
0     Facility not vermin proof. Harborage or condit...     A  12.0  
1     Facility not vermin proof. Harborage or condit...     A   9.0  
2     Plumbing not properly installed or maintained;...     A  13.0  
3     Live roaches present in facility's food and/or...     A  10.0  
4     Covered garbage receptacle not provided or ina...     A   9.0

6) Уберём «служебные» уровни: применим `droplevel` к столбцовому `MultiIndex` и переименуем имя уровня в `None`.  

In [75]:
insp_tidy.columns = insp_tidy.columns.droplevel(0).rename(None)
insp_tidy.head()

,Name,Date,Borough,Cuisine,Description,Grade,Score
0,3 STAR JUICE CENTER,2017-05-10,BROOKLYN,"Juice, Smoothies, Fruit Salads",Facility not vermin proof. Harborage or condit...,A,12.0
1,A & L PIZZA RESTAURANT,2017-08-22,BROOKLYN,Pizza,Facility not vermin proof. Harborage or condit...,A,9.0
2,AKSARAY TURKISH CAFE AND RESTAURANT,2017-07-25,BROOKLYN,Turkish,Plumbing not properly installed or maintained;...,A,13.0
3,ANTOJITOS DELI FOOD,2017-06-01,BROOKLYN,"Latin (Cuban, Dominican, Puerto Rican, South &...",Live roaches present in facility's food and/or...,A,10.0
4,BANGIA,2017-06-16,MANHATTAN,Korean,Covered garbage receptacle not provided or ina...,A,9.0


7) Этого `MultiIndex` можно было избежать, если преобразовать одно-столбцовый `DataFrame` в `Series` методом `squeeze` перед `unstack`.

In [76]:
inspections.set_index(['Name','Date', 'Info']) \
           .squeeze() \
           .unstack('Info') \
           .reset_index() \
           .rename_axis(None, axis='columns')

,Name,Date,Borough,Cuisine,Description,Grade,Score
0,3 STAR JUICE CENTER,2017-05-10,BROOKLYN,"Juice, Smoothies, Fruit Salads",Facility not vermin proof. Harborage or condit...,A,12.0
1,A & L PIZZA RESTAURANT,2017-08-22,BROOKLYN,Pizza,Facility not vermin proof. Harborage or condit...,A,9.0
2,AKSARAY TURKISH CAFE AND RESTAURANT,2017-07-25,BROOKLYN,Turkish,Plumbing not properly installed or maintained;...,A,13.0
3,ANTOJITOS DELI FOOD,2017-06-01,BROOKLYN,"Latin (Cuban, Dominican, Puerto Rican, South &...",Live roaches present in facility's food and/or...,A,10.0
4,BANGIA,2017-06-16,MANHATTAN,Korean,Covered garbage receptacle not provided or ina...,A,9.0
...,...,...,...,...,...,...,...
95,VALL'S PIZZERIA,2017-03-15,STATEN ISLAND,Pizza/Italian,Wiping cloths soiled or not stored in sanitizi...,A,9.0
96,VIP GRILL,2017-06-12,BROOKLYN,Jewish/Kosher,Hot food item not held at or above 140Âº F.,A,10.0
97,WAHIZZA,2017-04-13,MANHATTAN,Pizza,"No facilities available to wash, rinse and san...",A,10.0
98,WANG MANDOO HOUSE,2017-08-29,QUEENS,Korean,Accurate thermometer not provided in refrigera...,A,12.0


### Справочная информация

В качестве альтернативы `pivot` можно использовать `unstack`, который работает с уровнями индекса. Сначала переносим и «перекладываемые», и «неперекладываемые» столбцы в индекс (`set_index`), затем применяем `unstack`. При `unstack` `DataFrame` `pandas` сохраняет исходное имя столбца (здесь это `Value`) и создаёт столбцовый `MultiIndex`. Далее `reset_index` возвращает «внешние» уровни индекса в столбцы; параметр `col_level=-1` позволяет разместить имена на нижнем уровне. Очищаем оставшийся `MultiIndex` через `droplevel` и снимаем имя уровня (`None`).

## 2.11 Приведение к аккуратной форме, когда в одной ячейке хранится два и более значений

Табличные данные двумерны, поэтому объём информации в одной ячейке ограничен. Иногда встречаются наборы, где в ячейке хранится несколько значений. Аккуратные данные допускают ровно одно значение в каждой ячейке. Чтобы исправить ситуацию, обычно нужно распарсить строку в несколько столбцов методами `str`.

### Постановка задачи

Преобразовать набор, где один столбец содержит в каждой ячейке несколько переменных. Разделить строки на отдельные столбцы, чтобы получить аккуратную форму.

### Ключевые этапы

1) Прочитаем набор и определим переменные.

In [77]:
cities = pd.read_csv('data/texas_cities.csv')
cities

,City,Geolocation
0,Houston,"29.7604° N, 95.3698° W"
1,Dallas,"32.7767° N, 96.7970° W"
2,Austin,"30.2672° N, 97.7431° W"


2) `City` корректен (одна величина). Столбец `Geolocation` содержит четыре переменные: широта, направление широты, долгота, направление долготы. Разделим `Geolocation` на четыре столбца с помощью `str.split` по подходящему шаблону.  

In [78]:
geolocations = cities.Geolocation.str.split(pat='. ', expand=True)
geolocations.columns = ['latitude', 'latitude direction', 'longitude', 'longitude direction']
geolocations

,latitude,latitude direction,longitude,longitude direction
0,29.7604,N,95.3698,W
1,32.7767,N,96.7970,W
2,30.2672,N,97.7431,W


3) Поскольку исходный тип `Geolocation` &ndash; `str`, новые столбцы тоже будут `str`. Преобразуем широту и долготу в `float`.

In [79]:
geolocations.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   latitude             3 non-null      str  
 1   latitude direction   3 non-null      str  
 2   longitude            3 non-null      str  
 3   longitude direction  3 non-null      str  
dtypes: str(4)
memory usage: 228.0 bytes


In [80]:
geolocations = geolocations.astype({'latitude': 'float', 'longitude': 'float'})
geolocations.dtypes

latitude               float64
latitude direction         str
longitude              float64
longitude direction        str
dtype: object

4) Объедините новые столбцы со столбцом `City`.

In [81]:
cities_tidy = pd.concat([cities['City'], geolocations], axis='columns')
cities_tidy

,City,latitude,latitude direction,longitude,longitude direction
0,Houston,29.7604,N,95.3698,W
1,Dallas,32.7767,N,96.7970,W
2,Austin,30.2672,N,97.7431,W


### Справочная информация

Мы выбрали разбиение на четыре столбца, но можно было оставить два числовых столбца (широта и долгота), используя знак для направлений. Простой путь &ndash; `str.split` с регулярным выражением, задающим позиции разбиения (например, «символ градуса + пробел», затем «запятая + пробел»). В итоге три разбиения &ndash; четыре столбца. Затем конвертируем типы (`to_numeric`/`astype`). Объединяем с исходным столбцом города.

## 2.12 Приведение к аккуратной форме, когда переменные хранятся в именах столбцов и в значениях

Сложный вариант неаккуратных данных &ndash; когда часть переменных хранится горизонтально (в именах столбцов), а часть &ndash; вертикально (в значениях). Часто это результат уже подготовленного сводного отчёта.

### Постановка задачи

Идентифицировать переменные, заданные вертикально и горизонтально, и привести данные к аккуратной форме методами `melt` и `pivot_table`.

### Ключевые этапы

1) Прочитаем набор `sensors` и определим переменные.

In [82]:
sensors = pd.read_csv('data/sensors.csv')
sensors

,Group,Property,2012,2013,2014,2015,2016
0,A,Pressure,928,873,814,973,870
1,A,Temperature,1026,1038,1009,1036,1042
2,A,Flow,819,806,861,882,856
3,B,Pressure,817,877,914,806,942
4,B,Temperature,1008,1041,1009,1002,1013
5,B,Flow,887,899,837,824,873


2) Корректно вертикально расположен только `Group`. Столбец `Property` имеет три уникальные переменные: `Pressure`, `Temperature`, `Flow`. Остальные столбцы (`2012`–`2016`) &ndash; это одна переменная «Год» (`Year`). Одним методом `DataFrame` такую структуру не перестроить. Сначала используем `melt`, чтобы перенести годы в отдельный столбец.

In [83]:
sensors.melt(id_vars=['Group', 'Property'], var_name='Year').head(6)

,Group,Property,Year,value
0,A,Pressure,2012,928
1,A,Temperature,2012,1026
2,A,Flow,2012,819
3,B,Pressure,2012,817
4,B,Temperature,2012,1008
5,B,Flow,2012,887


3) Затем используем `pivot_table`, чтобы значения `Property` стали именами столбцов.

In [84]:
sensors.melt(id_vars=['Group', 'Property'], var_name='Year') \
       .pivot_table(index=['Group', 'Year'], columns='Property', values='value') \
       .reset_index() \
       .rename_axis(None, axis='columns')

,Group,Year,Flow,Pressure,Temperature
0,A,2012,819.0,928.0,1026.0
1,A,2013,806.0,873.0,1038.0
2,A,2014,861.0,814.0,1009.0
3,A,2015,882.0,973.0,1036.0
4,A,2016,856.0,870.0,1042.0
5,B,2012,887.0,817.0,1008.0
6,B,2013,899.0,877.0,1041.0
7,B,2014,837.0,914.0,1009.0
8,B,2015,824.0,806.0,1002.0
9,B,2016,873.0,942.0,1013.0


### Справочная информация

`Pandas` не умеет одновременно «поворачивать» несколько наборов столбцов, поэтому действуем по шагам. Сначала исправляем годы (`melt`, оставляя `Property` в `id_vars`). Полученный результат соответствует шаблону, где несколько переменных хранятся в значениях &ndash; для «поворота» используем `pivot_table` (можно указать несколько столбцов в `index`). После «поворота» переменные `Group` и `Year` оказываются в индексе &ndash; вернём их в столбцы (`reset_index`). Имя уровня столбцов, унаследованное от `columns`, после `reset_index` становится лишним &ndash; удалим его (`rename_axis(None, axis='columns')`).

### Дополнительно

Почти всегда есть альтернативный путь через `stack`/`unstack`: сначала перенесите столбцы, которые не «поворачиваются» сейчас, в индекс, затем применяйте нужную пару методов.

In [85]:
sensors.set_index(['Group', 'Property']) \
       .stack() \
       .unstack('Property') \
       .rename_axis(['Group', 'Year'], axis='index') \
       .rename_axis(None, axis='columns') \
       .reset_index()

,Group,Year,Flow,Pressure,Temperature
0,A,2012,819,928,1026
1,A,2013,806,873,1038
2,A,2014,861,814,1009
3,A,2015,882,973,1036
4,A,2016,856,870,1042
5,B,2012,887,817,1008
6,B,2013,899,877,1041
7,B,2014,837,914,1009
8,B,2015,824,806,1002
9,B,2016,873,942,1013


## 2.13 Приведение к аккуратной форме, когда в одной таблице хранятся несколько типов наблюдений

Данные проще сопровождать, когда каждая таблица содержит информацию только об одной наблюдательной единице. С другой стороны, для анализа иногда удобнее иметь всё в одной таблице, а для машинного обучения &ndash; тем более. Цель подхода `tidy` &ndash; не непосредственный анализ, а такая структура, которая упрощает последующий анализ. Если в одной таблице присутствуют несколько типов наблюдений, их может понадобиться разделить на отдельные таблицы.

### Постановка задачи

В наборе `movie` выделить три типа наблюдений (*фильмы*, *актёры*, *режиссёры*) и создать отдельные таблицы. Важно понимать, что число лайков актёров и режиссёров в социальной сети **независимо** от фильма: каждому актёру/режиссёру сопоставлено одно значение лайков. Благодаря этой независимости данные можно разделить. В реляционных БД такой процесс называется нормализацией: он повышает целостность данных и снижает избыточность.

### Ключевые этапы

1) Прочитаем модифицированный набор `movie` и вывести первые строки.  

In [86]:
movie = pd.read_csv('data/movie_altered.csv')
movie.head()

,title,rating,year,duration,director_1,director_social_likes_1,actor_1,actor_2,actor_3,actor_social_likes_1,actor_social_likes_2,actor_social_likes_3
0,Avatar,PG-13,2009.0,178.0,James Cameron,0.0,CCH Pounder,Joel David Moore,Wes Studi,1000.0,936.0,855.0
1,Pirates of the Caribbean: At World's End,PG-13,2007.0,169.0,Gore Verbinski,563.0,Johnny Depp,Orlando Bloom,Jack Davenport,40000.0,5000.0,1000.0
2,Spectre,PG-13,2015.0,148.0,Sam Mendes,0.0,Christoph Waltz,Rory Kinnear,Stephanie Sigman,11000.0,393.0,161.0
3,The Dark Knight Rises,PG-13,2012.0,164.0,Christopher Nolan,22000.0,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,27000.0,23000.0,23000.0
4,Star Wars: Episode VII - The Force Awakens,NaN,NaN,NaN,Doug Walker,131.0,Doug Walker,Rob Walker,NaN,131.0,12.0,NaN


2) Набор содержит сведения о фильме, режиссёре и актёрах &ndash; это три типа наблюдений. Сначала создадим уникальный идентификатор фильма (`insert` столбца `id`).  

In [87]:
movie.insert(0, 'id', np.arange(len(movie)))
movie.head()

,id,title,rating,year,duration,director_1,director_social_likes_1,actor_1,actor_2,actor_3,actor_social_likes_1,actor_social_likes_2,actor_social_likes_3
0,0,Avatar,PG-13,2009.0,178.0,James Cameron,0.0,CCH Pounder,Joel David Moore,Wes Studi,1000.0,936.0,855.0
1,1,Pirates of the Caribbean: At World's End,PG-13,2007.0,169.0,Gore Verbinski,563.0,Johnny Depp,Orlando Bloom,Jack Davenport,40000.0,5000.0,1000.0
2,2,Spectre,PG-13,2015.0,148.0,Sam Mendes,0.0,Christoph Waltz,Rory Kinnear,Stephanie Sigman,11000.0,393.0,161.0
3,3,The Dark Knight Rises,PG-13,2012.0,164.0,Christopher Nolan,22000.0,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,27000.0,23000.0,23000.0
4,4,Star Wars: Episode VII - The Force Awakens,NaN,NaN,NaN,Doug Walker,131.0,Doug Walker,Rob Walker,NaN,131.0,12.0,NaN


3) Используем `wide_to_long`, чтобы собрать всех актёров в один столбец, их лайки в социальной сети &ndash; в другой; аналогично для режиссёра (хотя он один на фильм).  

In [88]:
stubnames = ['director', 'director_social_likes', 'actor', 'actor_social_likes']
movie_long = pd.wide_to_long(movie, 
                             stubnames=stubnames, 
                             i='id', 
                             j='num', 
                             sep='_').reset_index()
movie_long['num'] = movie_long['num'].astype(int)
movie_long.head(9)

,id,num,duration,rating,title,year,director,director_social_likes,actor,actor_social_likes
0,0,1,178.0,PG-13,Avatar,2009.0,James Cameron,0.0,CCH Pounder,1000.0
1,1,1,169.0,PG-13,Pirates of the Caribbean: At World's End,2007.0,Gore Verbinski,563.0,Johnny Depp,40000.0
2,2,1,148.0,PG-13,Spectre,2015.0,Sam Mendes,0.0,Christoph Waltz,11000.0
3,3,1,164.0,PG-13,The Dark Knight Rises,2012.0,Christopher Nolan,22000.0,Tom Hardy,27000.0
4,4,1,NaN,NaN,Star Wars: Episode VII - The Force Awakens,NaN,Doug Walker,131.0,Doug Walker,131.0
5,5,1,132.0,PG-13,John Carter,2012.0,Andrew Stanton,475.0,Daryl Sabara,640.0
6,6,1,156.0,PG-13,Spider-Man 3,2007.0,Sam Raimi,0.0,J.K. Simmons,24000.0
7,7,1,100.0,PG,Tangled,2010.0,Nathan Greno,15.0,Brad Garrett,799.0
8,8,1,141.0,PG-13,Avengers: Age of Ultron,2015.0,Joss Whedon,0.0,Chris Hemsworth,26000.0


4) Теперь данные готовы к разбиению на несколько меньших таблиц (отдельно *фильмы*, *актёры*, *режиссёры*), сохраняя `id` и служебный номер `num` исходной позиции.  

In [89]:
movie_table = movie_long[['id','title', 'year', 'duration', 'rating']]
director_table = movie_long[['id', 'director', 'num', 'director_social_likes']]
actor_table = movie_long[['id', 'actor', 'num', 'actor_social_likes']]

In [90]:
movie_table.head(9)

,id,title,year,duration,rating
0,0,Avatar,2009.0,178.0,PG-13
1,1,Pirates of the Caribbean: At World's End,2007.0,169.0,PG-13
2,2,Spectre,2015.0,148.0,PG-13
3,3,The Dark Knight Rises,2012.0,164.0,PG-13
4,4,Star Wars: Episode VII - The Force Awakens,NaN,NaN,NaN
5,5,John Carter,2012.0,132.0,PG-13
6,6,Spider-Man 3,2007.0,156.0,PG-13
7,7,Tangled,2010.0,100.0,PG
8,8,Avengers: Age of Ultron,2015.0,141.0,PG-13


In [91]:
director_table.head(9)

,id,director,num,director_social_likes
0,0,James Cameron,1,0.0
1,1,Gore Verbinski,1,563.0
2,2,Sam Mendes,1,0.0
3,3,Christopher Nolan,1,22000.0
4,4,Doug Walker,1,131.0
5,5,Andrew Stanton,1,475.0
6,6,Sam Raimi,1,0.0
7,7,Nathan Greno,1,15.0
8,8,Joss Whedon,1,0.0


In [92]:
actor_table.head(9)

,id,actor,num,actor_social_likes
0,0,CCH Pounder,1,1000.0
1,1,Johnny Depp,1,40000.0
2,2,Christoph Waltz,1,11000.0
3,3,Tom Hardy,1,27000.0
4,4,Doug Walker,1,131.0
5,5,Daryl Sabara,1,640.0
6,6,J.K. Simmons,1,24000.0
7,7,Brad Garrett,1,799.0
8,8,Chris Hemsworth,1,26000.0
